# Plaque assay

**What it does.** Detect and measure plaques, reporting count and size per well.

**When to use it.** For lytic-cycle experiments where the readout is how much monolayer was destroyed.

**What you get.** Per-plaque measurements and a per-well summary.

---

> Every path below is a placeholder. Point `src` at your own data before running.
> Nothing in this notebook writes outside the folder you give it.

## 1. Check the install

If this cell fails, the rest cannot work. It reports the version and whether a GPU is visible — segmentation and training are usable on CPU but slow.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. The function this notebook runs

`spacr.submodules.analyze_plaques`

```
analyze_plaques(settings)
```

Segment host-cell plaques with a bundled Cellpose model and summarize per-image counts and areas.

In [ ]:
from spacr.submodules import analyze_plaques

## 3. Settings

`spacr.settings.get_analyze_plaque_settings` fills in every default, so you only have to write down what differs. The cell below prints the full set as it exists in this version — treat that output as the reference, not this notebook.

Change values in `settings`, not in the defaults helper.

In [ ]:
from spacr.settings import get_analyze_plaque_settings

defaults = get_analyze_plaque_settings({})
for key in sorted(defaults):
    print(f'{key:38s} {defaults[key]!r}')

### Every setting this function accepts

The full dictionary, each key on its own line with its default and what it controls. Edit values in place; delete nothing — a key left at its default behaves exactly as if it were absent.

Generated from this installed version, so it is the real set of keys, the real defaults and the real descriptions.

In [ ]:
settings = {
    # (float) - Cellpose cellprob_threshold: the cell-probability
    # cut-off applied to the network output when deciding which pixels
    # belong to an object. Lower it (typically toward -6) to recover dim
    # or partly detected objects and grow existing masks; raise it
    # (toward 6) to drop faint false positives and shrink masks. Default
    # 0.
    'CP_prob': 0,

    # (int) - Multiplier on background that sets the signal threshold
    # (background * Signal_to_noise) used by the Cellpose tools when
    # normalizing images: for each channel of each image spaCR takes the
    # first of the 98th, 99th, 99.9th, 99.99th and 99.999th percentiles
    # whose value exceeds that threshold, averages the picks over the
    # batch, and rescales the channel between its average 2nd percentile
    # and that upper bound. Raise it to force a higher percentile - less
    # clipping, dimmer output; lower it to stretch faint objects harder
    # at the cost of saturating bright ones. If no percentile clears the
    # threshold for a channel, its upper bound falls back to that
    # channel's average 2nd percentile, collapsing the range - a sign
    # the value is far too high. Ignored when percentiles is set.
    # Default 10 (5 in check_cellpose_models).
    'Signal_to_noise': 10,

    # (float) - Per-channel background level in raw intensity units.
    # Pixels below it are zeroed when remove_background is on, and it is
    # multiplied by Signal_to_noise to set the upper anchor for
    # normalization. Raise it if faint haze survives; set it too high
    # and dim real objects vanish. Default 100 (200 for Cellpose
    # training and plaque analysis).
    'background': 200,

    # (int) - How many images are held and processed together in one
    # pass: field stacks during normalization and Cellpose segmentation,
    # crops per step during classifier training and activation maps.
    # Raising it speeds runs up but increases RAM/VRAM roughly linearly;
    # lower it on out-of-memory errors. Defaults: 50 for mask
    # generation, 64 for training.
    'batch_size': 50,

    # (float) - (DEPRECEATED) Expected object diameter in pixels passed
    # as model.eval(diameter=...) by the mask-finetune tool and by
    # check_cellpose_models; Cellpose resizes each image by 30/diameter
    # so objects match the network's ~30 px working size, so a value
    # below the true size upscales the image and a value above
    # downscales it. It is also handed to CellposeModel as diam_mean
    # when custom_model is set, which the installed Cellpose 4.x ignores
    # with a warning, and it seeds the diameter spinbox of the Qt live
    # preview. It does not feed the segmentation pipeline's per-object
    # diameters or its minimum/maximum size filters - those are built
    # from magnification and cell_/nucleus_/pathogen_diameter. Default
    # 30 (40 in check_cellpose_models).
    'diameter': 30,

    # (bool) - Post-process each Cellpose mask in the mask-finetune /
    # plaque tool with fill_holes_in_mask: the mask is first re-labelled
    # by connectivity over all non-zero pixels (scipy.ndimage.label),
    # then interior holes are filled component by component. Because
    # re-labelling ignores the original label values, objects that touch
    # are merged into a single object and every object is renumbered
    # 1..n, so enable it when punched-out interiors (dark vacuoles,
    # nuclei inside a cell) should count as object area and object
    # identity does not matter, and disable it when touching objects
    # must stay distinct. Default True.
    'fill_in': True,

    # (float) - Cellpose flow_threshold: the maximum allowed error
    # between the predicted flow field and the flows recomputed from
    # each candidate mask; masks above it are discarded. Raise it to
    # keep more objects, including irregularly shaped ones; lower it to
    # reject poorly formed masks and reduce false positives. Default
    # 0.4.
    'flow_threshold': 0.4,

    # (bool) - Run Cellpose segmentation for every object channel you
    # defined (cell, nucleus, pathogen, organelle) and write label
    # stacks to masks/<object>_mask_stack. Set False to do preprocessing
    # only - build the normalized arrays now and segment later - but
    # Measure will then have nothing to quantify. Default True.
    'masks': True,

    # (bool) - Passed to Cellpose model.eval: run the mask-tracking
    # dynamics at full image resolution instead of on the downsampled
    # network grid. Enabling it gives smoother, better-fitting object
    # outlines at the cost of time and memory, and helps most when
    # objects differ a lot from the model's training diameter. Default
    # False; the object pipeline sets True for cell/nucleus and False
    # for pathogen.
    'resample': False,

    # (float) - Rescaling factor for the images.
    'rescale': False,

    # (bool or float) - Resize every image to target_height x
    # target_width before running Cellpose, then scale the returned mask
    # back to the original dimensions with nearest-neighbour
    # interpolation so measurements stay in original pixels. Turn it on
    # to bring oversized fields to the scale a model was trained at, or
    # to cut GPU memory. Requires target_height and target_width.
    # Default False (True for plaque analysis).
    'resize': True,

    # (bool or list of bool) - Whether to save masks to disk. Can be a
    # list of three booleans for [cell, nucleus, pathogen]
    # independently.
    'save': True,

    # (str, path) - Folder the current step reads from and writes into:
    # raw images for mask generation, the merged/ folder of .npy stacks
    # for measure, the plate root for dataset/regression steps, or the
    # folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/,
    # measurements/measurements.db, datasets/, results/) are created
    # inside it. A list of paths, or a "['a','b']" string, processes
    # several plates in one run.
    'src': 'path',

    # (int) - Height in pixels that images are resized to before
    # segmentation; masks are scaled back to the original dimensions
    # afterwards. Only applied when both target_height and target_width
    # are set (and, on the non-normalized path, when resize is True).
    # Use it to match the field size the model was trained at. Default
    # None, which disables resizing; 1120 for plaque analysis.
    'target_height': 1120,

    # (int) - Width in pixels that images are resized to before
    # segmentation; masks are scaled back to the original dimensions
    # afterwards. Only applied when both target_width and target_height
    # are set (and, on the non-normalized path, when resize is True).
    # Use it to match the field size the model was trained at. Default
    # None, which disables resizing; 1120 for plaque analysis.
    'target_width': 1120,

    # (bool) - Print extra run detail instead of the minimal log: the
    # resolved settings table at the start of mask generation, the
    # channel and Cellpose-model choices per object type, per-table row
    # counts and how many objects survive the nuclei/pathogen-per-cell
    # filters when measurement tables are merged, and extra
    # loader/diagnostic output in the training and UMAP paths. It only
    # adds console output, so turn it on when object counts come out
    # unexpected and you need to see which stage removed them. Defaults
    # are per-pipeline: True for mask generation, UMAP, screen analysis,
    # barcode mapping, Cellpose training and plaque analysis; False for
    # measure-and-crop, plot-from-db and plot-from-CSV, the endodyogeny
    # and class-proportion helpers, the Cellpose check/finetune tools,
    # and the screen regression, whose verbose branch display()s the
    # whole per-object score table.
    'verbose': True,

}

# Fill in anything left unset, then check the source path.
settings = get_analyze_plaque_settings(settings)
settings['src']

## 4. Run it

This is the long cell. Progress is logged; if you want more of it, raise the log levels in Preferences → Logging, or set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter.

In [ ]:
analyze_plaques(settings)

## Where the output went

Per-plaque measurements and a per-well summary.

spaCR writes beside the source folder rather than into a global location, so a plate stays self-contained and re-running does not clobber a different experiment.

### Next steps

* The GUI covers the same workflows with the settings laid out as a form — `python -m spacr`.
* The narrated walkthroughs are at <https://einarolafsson.github.io/spacr/tutorials/>.
* The API reference is at <https://einarolafsson.github.io/spacr/>.